In [7]:
API_TOKEN = "CWB-B1034BF8-0855-45E6-8F10-7C7D4DB194AA" 

In [8]:
import requests
import json


LAT_NORTH = 25.30
LAT_SOUTH = 24.75
LON_WEST = 120.90
LON_EAST = 122.00

filtered_stations = []
api_sources = ["O-A0001-001", "O-A0002-001"]


for source in api_sources:
    url = f"https://opendata.cwa.gov.tw/api/v1/rest/datastore/{source}?Authorization={API_TOKEN}"
    try:
        response = requests.get(url)
        if response.status_code == 200:
            data = response.json()
            records = data.get('records', {})
            stations = records.get('Station', [])
            
            for site in stations:
                try:
                    coordinates_list = site['GeoInfo']['Coordinates']
                    lon, lat = None, None
                    for coord in coordinates_list:
                        if coord.get('CoordinateName') == 'WGS84':
                            lon = float(coord['StationLongitude'])
                            lat = float(coord['StationLatitude'])
                            break
                    if lon is None or lat is None:
                        lon = float(coordinates_list[0]['StationLongitude'])
                        lat = float(coordinates_list[0]['StationLatitude'])
                    
                    station_id = site['StationId']
                    station_name = site['StationName']
                    
                    # 🌟 依據 ID 開頭與官方規範進行「四色測站」邏輯分類
                    station_type = "未知站"
                    if station_id.startswith("46"):
                        station_type = "有人站"
                    elif station_id.startswith("C0"):
                        station_type = "自動站"
                    elif station_id.startswith("C1"):
                        station_type = "雨量站"
                    elif station_id.startswith("C2"):
                        station_type = "農業站"
                    else:
                        # 防呆：有些新測站代號不同，若由 O-A0001 傳統大站出來的皆視為有人站
                        if source == "O-A0001-001":
                            station_type = "有人站"
                        else:
                            station_type = "自動站"
                    
                    if (LAT_SOUTH <= lat <= LAT_NORTH) and (LON_WEST <= lon <= LON_EAST):
                        filtered_stations.append({
                            "id": station_id,
                            "name": station_name,
                            "lat": lat,
                            "lon": lon,
                            "type": station_type  # 儲存分類標籤
                        })
                except Exception:
                    continue
            print(f"➔ {source} 處理完成。")
    except Exception as e:
        print(f"處理 {source} 時發生異常: {e}")

output_filename = "stations.json"
with open(output_filename, 'w', encoding='utf-8') as f:
    json.dump(filtered_stations, f, ensure_ascii=False, indent=2)

print(f"\nTotal {len(filtered_stations)} stn.")

➔ O-A0001-001 處理完成。
➔ O-A0002-001 處理完成。

Total 428 stn.
